<a href="https://colab.research.google.com/github/rehndjytuen/Face-Recognition-Project-DLS-2026/blob/main/%D0%9E%D1%81%D0%BD%D0%BE%D0%B2%D0%BD%D0%BE%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_2_(ArcFace_Loss).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# План заданий

Итак, вот, что от вас требуется в этом задании:

* Выбрать модель (или несколько моделей) для обучения. Можно брать предобученные на ImageNet, но нельзя использовать модели, предобученные на задачу распознавания лиц.
* Обучить эту модель (модели) на CE loss. Добиться accuracy > 0.7.
* Реализовать ArcFace loss.
* Обучить модель (модели) на ArcFace loss. Добиться accuracy > 0.7.
* Написать небольшой отчет по обучению, сравнить CE loss и ArcFace loss.

**P.S. Не забывайте сохранять модели после обучения**

делим наш выровненный датасет на обучающий, валидационный и тестовый. Отслеживаем, чтобы фото каждой "уникальной личности" попали в каждый из наборов.

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.colab import drive

# монтирую диск
drive.mount('/content/drive')

# пути
base_path = '/content/drive/MyDrive/celeba_project'
aligned_output_dir_new = os.path.join(base_path, 'rotated_aligned_dataset_NEW')
processed_landmarks_path_new = os.path.join(base_path, 'processed_landmarks_NEW.csv')
identity_file = os.path.join(base_path, 'annotations/identity_CelebA.txt')

split_dataset_base_path_overlap = os.path.join(base_path, 'split_aligned_dataset_OVERLAP')
os.makedirs(split_dataset_base_path_overlap, exist_ok=True)

train_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'train_images_OVERLAP')
val_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'val_images_OVERLAP')
test_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'test_images_OVERLAP')

os.makedirs(train_dir_overlap, exist_ok=True)
os.makedirs(val_dir_overlap, exist_ok=True)
os.makedirs(test_dir_overlap, exist_ok=True)

print(f"Created directories for OVERLAP split dataset at: {split_dataset_base_path_overlap}")

processed_landmarks_df_new = pd.read_csv(processed_landmarks_path_new)
identity_df = pd.read_csv(identity_file, sep=r'\s+', header=None, names=['image_id', 'person_id'])
# объединяим ландмарки и идентификаторы персон, чтобы получить person_id для каждого изображения в нашем новом наборе данных.
merged_df_overlap = pd.merge(processed_landmarks_df_new, identity_df, on='image_id', how='left')

# получаем список всех идентификаторов изображений из нашего датасета
all_image_ids = merged_df_overlap['image_id'].tolist()

# делим сперва на train и temp(validation + test)
train_images_overlap, temp_images_overlap = train_test_split(all_image_ids, test_size=0.3, random_state=42)

# делим temp на validation и test
val_images_overlap, test_images_overlap = train_test_split(temp_images_overlap, test_size=0.5, random_state=42)

print(f"Total images: {len(all_image_ids)}")
print(f"Train images: {len(train_images_overlap)}")
print(f"Validation images: {len(val_images_overlap)}")
print(f"Test images: {len(test_images_overlap)}")

# функция для копирования изображений и соответствующих им данных
def distribute_dataset_overlap(image_list, target_img_dir, target_meta_file, source_df, source_img_dir):
    print(f"Copying {len(image_list)} images to {target_img_dir}")
    current_meta_data = []
    for img_name in tqdm(image_list):
        src_img_path = os.path.join(source_img_dir, img_name)
        dst_img_path = os.path.join(target_img_dir, img_name)

        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
            current_meta_data.append(source_df[source_df['image_id'] == img_name])
        else:
            print(f"Warning: Image {img_name} not found in {source_img_dir}. Skipping.")

    if current_meta_data:
        current_meta_df = pd.concat(current_meta_data)
        current_meta_df.to_csv(target_meta_file, index=False)
        print(f"Saved metadata for {len(current_meta_df)} images to {target_meta_file}")
    else:
        print(f"No metadata to save for {target_img_dir}")

distribute_dataset_overlap(train_images_overlap, train_dir_overlap, os.path.join(split_dataset_base_path_overlap, 'train_meta_OVERLAP.csv'), merged_df_overlap, aligned_output_dir_new)
distribute_dataset_overlap(val_images_overlap, val_dir_overlap, os.path.join(split_dataset_base_path_overlap, 'val_meta_OVERLAP.csv'), merged_df_overlap, aligned_output_dir_new)
distribute_dataset_overlap(test_images_overlap, test_dir_overlap, os.path.join(split_dataset_base_path_overlap, 'test_meta_OVERLAP.csv'), merged_df_overlap, aligned_output_dir_new)

print("OVERLAP Dataset splitting and distribution complete!")

Created directories for OVERLAP split dataset at: /content/drive/MyDrive/celeba_project/split_aligned_dataset_OVERLAP
Total images: 10500
Train images: 7350
Validation images: 1575
Test images: 1575
Copying 7350 images to /content/drive/MyDrive/celeba_project/split_aligned_dataset_OVERLAP/train_images_OVERLAP...


100%|██████████| 7350/7350 [42:17<00:00,  2.90it/s]


Saved metadata for 7350 images to /content/drive/MyDrive/celeba_project/split_aligned_dataset_OVERLAP/train_meta_OVERLAP.csv
Copying 1575 images to /content/drive/MyDrive/celeba_project/split_aligned_dataset_OVERLAP/val_images_OVERLAP...


100%|██████████| 1575/1575 [09:06<00:00,  2.88it/s]


Saved metadata for 1575 images to /content/drive/MyDrive/celeba_project/split_aligned_dataset_OVERLAP/val_meta_OVERLAP.csv
Copying 1575 images to /content/drive/MyDrive/celeba_project/split_aligned_dataset_OVERLAP/test_images_OVERLAP...


100%|██████████| 1575/1575 [09:02<00:00,  2.90it/s]


Saved metadata for 1575 images to /content/drive/MyDrive/celeba_project/split_aligned_dataset_OVERLAP/test_meta_OVERLAP.csv
OVERLAP Dataset splitting and distribution complete!


## Выбор модели для задачи распознавания лиц

Для задачи распознавания лиц, учитывая ограничения (можно использовать модели, предобученные на ImageNet, но нельзя использовать модели, предобученные на задачу распознавания лиц), мой китайский друг дипсик ответственно заявил, что отличным выбором будет модель семейства **ResNet**, например, **ResNet50**.

**Обоснование:**

*   **ImageNet Pre-training:** ResNet50, предобученный на ImageNet, является мощным экстрактором признаков общего назначения, что позволяет получить богатые иерархические представления изображений.
*   **Избегание предвзятости:** Поскольку модель не была обучена специально для задач распознавания лиц, она не содержит в себе специфических для лица весов, что соответствует вашему требованию и позволяет построить собственную систему распознавания.
*   **Производительность:** ResNet50 предлагает хороший баланс между глубиной сети, количеством параметров и производительностью, что делает его подходящим для многих задач компьютерного зрения.

Будем использовать ResNet50 как backbone для извлечения признаков, к которому можно будет добавить свою "голову" (например, полносвязные слои) для классификации или вычисления эмбеддингов лиц.

Попробуем обучить ResNet50 сперва на основе CE loss, как требуется в перыой части задания

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import pandas as pd
from sklearn.metrics import accuracy_score
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# пути наши ...
base_path = '/content/drive/MyDrive/celeba_project'
split_dataset_base_path_overlap = os.path.join(base_path, 'split_aligned_dataset_OVERLAP')

train_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'train_images_OVERLAP')
val_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'val_images_OVERLAP')
test_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'test_images_OVERLAP')

train_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'train_meta_OVERLAP.csv'))
val_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'val_meta_OVERLAP.csv'))
test_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'test_meta_OVERLAP.csv'))

all_person_ids_overlap = pd.concat([train_meta_df_overlap['person_id'], val_meta_df_overlap['person_id'], test_meta_df_overlap['person_id']]).unique()
person_to_label_map_overlap = {person_id: i for i, person_id in enumerate(all_person_ids_overlap)}
num_classes_overlap = len(person_to_label_map_overlap)

print(f"Number of unique persons (classes) in OVERLAP split: {num_classes_overlap}")

transform = transforms.Compose([
    transforms.Resize((224, 224)), # под ResNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FaceRecognitionDataset(Dataset):
    def __init__(self, img_dir, identity_df, person_to_label_map, transform=None):
        self.img_dir = img_dir
        self.identity_df = identity_df.copy()
        self.transform = transform
        self.person_to_label_map = person_to_label_map

        self.identity_df['label'] = self.identity_df['person_id'].map(self.person_to_label_map)
        self.image_names = self.identity_df['image_id'].tolist()
        self.labels = self.identity_df['label'].tolist()

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

class FaceRecognitionModel(nn.Module):
    def __init__(self, backbone, embedding_dim=512, num_classes=None):
        super().__init__()
        self.backbone = backbone

        backbone_out_features = 2048
        if hasattr(backbone, 'fc'):
            backbone_out_features = backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif hasattr(backbone, 'classifier'):
            in_features = backbone.classifier[-1].in_features
            backbone.classifier = nn.Identity()
        else:
             pass

        self.embedding = nn.Sequential(
            nn.Linear(backbone_out_features, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True)
        )

        if num_classes is None:
            raise ValueError("num_classes must be provided for the classification head.")
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, return_embedding=False):
        features = self.backbone(x)

        if features.dim() > 2:
            features = features.view(features.size(0), -1)

        emb = self.embedding(features)
        if return_embedding:
            return emb
        logits = self.classifier(emb)
        return logits

# наши датасеты
train_dataset_overlap = FaceRecognitionDataset(
    img_dir=train_img_dir_overlap,
    identity_df=train_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

val_dataset_overlap = FaceRecognitionDataset(
    img_dir=val_img_dir_overlap,
    identity_df=val_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

test_dataset_overlap = FaceRecognitionDataset(
    img_dir=test_img_dir_overlap,
    identity_df=test_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)


batch_size = 32
train_loader_overlap = DataLoader(train_dataset_overlap, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader_overlap = DataLoader(val_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader_overlap = DataLoader(test_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train (OVERLAP) dataset size: {len(train_dataset_overlap)} images")
print(f"Validation (OVERLAP) dataset size: {len(val_dataset_overlap)} images")
print(f"Test (OVERLAP) dataset size: {len(test_dataset_overlap)} images")

model_ce_overlap = FaceRecognitionModel(num_classes_overlap, model_resnet50).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_ce_overlap.parameters(), lr=1e-4)
num_epochs = 10 # для начала

best_val_accuracy_overlap = 0.0
model_save_path_ce_overlap = os.path.join(base_path, 'face_recognition_resnet50_ce_overlap.pth')

print("\nStarting training with Cross-Entropy Loss on OVERLAP dataset...")

for epoch in range(num_epochs):
    model_ce_overlap.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in tqdm(train_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Train OVERLAP)"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_ce_overlap(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader_overlap)
    train_accuracy = correct_train / total_train

    # validation
    model_ce_overlap.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Validation OVERLAP)"):
            images, labels = images.to(device), labels.to(device)
            outputs = model_ce_overlap(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss /= len(val_loader_overlap)
    val_accuracy = correct_val / total_val

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

    # сохраняем лучшую модель
    if val_accuracy > best_val_accuracy_overlap:
        best_val_accuracy_overlap = val_accuracy
        torch.save(model_ce_overlap.state_dict(), model_save_path_ce_overlap)
        print(f"Model saved! Best validation accuracy: {best_val_accuracy_overlap:.4f}")

print("\nTraining with Cross-Entropy Loss on OVERLAP dataset finished.")
print(f"Best validation accuracy achieved: {best_val_accuracy_overlap:.4f}")
print(f"Model with best validation accuracy saved to: {model_save_path_ce_overlap}")

# test set
model_ce_overlap.load_state_dict(torch.load(model_save_path_ce_overlap))
model_ce_overlap.eval()

correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader_overlap, desc="Evaluating on Test Set (OVERLAP)"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_ce_overlap(images)
        _, predicted = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_accuracy_overlap = correct_test / total_test
print(f"\nTest Accuracy with Cross-Entropy Loss (OVERLAP dataset): {test_accuracy_overlap:.4f}")

Number of unique persons (classes) in OVERLAP split: 350
Train (OVERLAP) dataset size: 7350 images
Validation (OVERLAP) dataset size: 1575 images
Test (OVERLAP) dataset size: 1575 images

Starting training with Cross-Entropy Loss on OVERLAP dataset...


Epoch 1/10 (Validation OVERLAP): 100%|██████████| 50/50 [00:12<00:00,  4.14it/s]


Epoch 1: Train Loss: 3.0152, Train Acc: 0.5465, Val Loss: 1.2602, Val Acc: 0.7949
Model saved! Best validation accuracy: 0.7949


Epoch 2/10 (Validation OVERLAP): 100%|██████████| 50/50 [00:12<00:00,  4.01it/s]


Epoch 2: Train Loss: 0.6974, Train Acc: 0.9158, Val Loss: 0.8097, Val Acc: 0.8305
Model saved! Best validation accuracy: 0.8305


...

Epoch 9: Train Loss: 0.0053, Train Acc: 1.0000, Val Loss: 0.2146, Val Acc: 0.9530


Epoch 10/10 (Validation OVERLAP): 100%|██████████| 50/50 [00:13<00:00,  3.78it/s]


Epoch 10: Train Loss: 0.0042, Train Acc: 1.0000, Val Loss: 0.2101, Val Acc: 0.9549

Training with Cross-Entropy Loss on OVERLAP dataset finished.
Best validation accuracy achieved: 0.9562
Model with best validation accuracy saved to: /content/drive/MyDrive/celeba_project/face_recognition_resnet50_ce_overlap.pth


Evaluating on Test Set (OVERLAP): 100%|██████████| 50/50 [00:13<00:00,  3.79it/s]


Test Accuracy with Cross-Entropy Loss (OVERLAP dataset): 0.9549


**Как устроен ArcFace**:

Стандартные SoftMax + кросс-энтропия (CE) выглядят так:

$$L_{CE} = \frac{-1}{N}\sum_1^N \frac{e^{W_{y_i}^{T}x_i + b_{y_i}}}{\sum^n_{j=1}e^{W_j^Tx_i+b_j}},$$

здесь:
- $x_i \in \mathbb{R^d}$ — вектор $i$-го элемента обучающей выборки перед последним полносвязным слоем сети. $y_i$ — класс этого элемента;
- $W_j \in \mathbb{R^d}$ — j-ый столбец матрицы весов последнего слоя сети (т.е. слоя, который производит итоговую классификацю входящего объекта);
- $b_j \in \mathbb{R^d}$ — j-ый элемент вектора байеса последнего слоя сети;
- $N$ — batch size;
- $n$ — количество классов.


Хотя этот лосс работает хорошо, он явным образом не заставляет эмбеддинги $x_i$ элементов, принадлежащих одному классу, быть близкими друг к другу по расстоянию. И не заставляет эмбеддинги элементов, принадлежащих разным классам, быть далеко друг от друга. Все, что хочет этот лосс — чтобы на основе эмбеддингов $x_i$ можно было хорошо классифицировать элементы, никакие ограничений на расстояния между эмбеддингами $x_i$ он не вводит.

Из-за этого у нейросетей для распознавания лиц, которые обучены на обычном CE loss, бывают проблемы с распознаванием лиц, которые сильно отличаются от лиц того же человека разными доп. атрибутами (шляпа/прическа/очки и т.п.). Просто эмбеддинг для таких лиц получается довольно далек по расстоянию от других эмбеддингов лиц этого же человека.

Давайте теперь немного поправим формулу:
- уберем байес последнего слоя, т.е. сделаем $b_j=0$;
- нормализуем веса последнего слоя: ||$W_j$|| = 1;
- нормализуем эмбеддинги: ||$x_i$|| = 1. Перед подачей их на вход последнему слою (т.е. перед умножением на матрицу $W_j$) умножим их на гиперпараметр s. По сути, мы приводим норму всех эмбеддингов к s. Смысл этого гиперпараметра в том, что, возможно, сети проще будет классифицировать эмбеддинги, у которых не единичная норма.

Нормализация приводит к тому, что эмбеддинги распределяются по сфере единичного радиуса (и сфере радиуса s после умножения на гиперпараметр s). И итоговые предсказания сети после последнего слоя зависят только от угла между эмбеддингами $x_i$ и выученных весов $W_j$. От нормы эмбеддинга $x_i$ они больше не зависят, т.к. у всех эмбеддингов они теперь одинаковые.

Получается, в степени экспоненты у нас останется выражение $s W_{y_i}^{T}x_i$, которое можно переписать в виде  $s W_{y_i}^{T}x_i = s ||W_{y_i}||\cdot ||x_i|| \cdot cos\Theta_{y_i}$. Тут $\Theta_{y_i}$ — это угол между векторами $W_{y_i}$ и $x_i$. Но так как мы сделали нормы $W_{y_i}$ и $x_i$ единичными, то все это выражение просто будет равно $s cos\Theta_{y_i}$.

В итоге мы получим следующую формулу лосса:

$$L = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos\Theta_{y_i}}}{e^{s\ cos\Theta_{y_i}} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$


И последний шаг. Добавим еще один гиперпараметр $m$. Он называется additive angular margin penalty и заставляет эмбеддинги одного класса быть ближе друг к другу, а эмбеддинги разных классов — более далекими друг от друга.

В итоге получим вот что:

$$L_{ArcFace} = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos(\Theta_{y_i} + m)}}{e^{s\ cos(\Theta_{y_i} + m)} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$

Это и есть ArcFace Loss с двумя  гиперпараметрами, s и m.

Получается, что ArcFace Loss завтавляет сеть выучивать эмбеддинги, распределенные по сфере радиуса s, причем чтобы эмбеддинги одного класса были ближе друг к другу, а эмбеддинги разных классов — более далеки друг от друга.

![ArcFace](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTKRR-YA_XR3yhIYBbkc8Zlbua0Q2WdM3gx_g&s)

Важное пояснение:

Строго говоря, ArcFace - не лосс, отдельный архитектурный модуль модификация SoftMax. Он реализует идею внесения геометрического отступа непосредственно в пространство признаков. Для обучения в качестве лосса используется обычная кросс-энтропия. Более конкретно по шагам:

Вы извлекаете эмбеддинги из бэкбона сети (предобученной модели, у которой обрезан FC-слой, если он был)
Эти эмбеддинги поступают в ArcFace-слой, который содержит векторы-центры для каждого класса (веса классификатора) и логику нормализации и добавления углового отступа
Для целевого класса ArcFace-слой преобразует косинус угла  θ  в  cos(θ+m)
Для остальных классов оставляет обычный косинус  cos(θ)
Эти модифицированные логиты подаются на вход стандартной функции Cross-Entropy
Градиенты от Cross-Entropy текут назад через ArcFace-слой к бэкбону, обучая модель извлекать эмбеддинги
Результат: модифицированные логиты с "жестким" разделением для целевого класса, а значит и более качественные эмбеддинги.

Схема:

[Изображение] → [Бэкбон] → [ЭМБЕДДИНГ] → [ArcFace] → [Логиты] → [CE Loss]
                    │                        │           │          
                   CNN                   Нормализация   Оценки
                                          + Angular    для всех
                                            Margin     классов
Для получения качественных эмбеддингов после обучения ArcFace-слой больше не нужен, и его обычно обрезают. Он нужен был только обучения модели, и поэтому часто ArcFace называю именно лоссом.

Итак, вот, что от вас требуется в этом задании:

Выбрать модель для обучения. Можно брать предобученные на ImageNet, но нельзя использовать модели, предобученные на задачу распознавания лиц.
Обучить эту модель (модели) на CE loss. Добиться accuracy > 0.7.
Реализовать ArcFace loss.
Обучить модель (модели) на ArcFace loss. Добиться accuracy > 0.7.
Написать небольшой отчет по обучению, сравнить CE loss и ArcFace loss.
P.S. Не забывайте сохранять модели после обучения

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import torchvision.models as models
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from google.colab import drive

# drive.mount('/content/drive')

class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)

    def forward(self, input_features, label):
        cosine = F.linear(F.normalize(input_features), F.normalize(self.weight))
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m

        one_hot = torch.zeros(cosine.size(), device=input_features.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)

        output_logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output_logits *= self.s
        return output_logits

# без внутреннего классификатора
class FaceRecognitionModelArcFace(nn.Module):
    def __init__(self, num_classes, backbone, embedding_dim=512, s=64.0, m=0.50):
        super().__init__()
        self.backbone = backbone
        backbone_out_features = 2048 # по умолчанию для ResNet50 после avgpool

        if hasattr(backbone, 'fc') and isinstance(backbone.fc, nn.Linear):
            backbone_out_features = backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif hasattr(backbone, 'classifier') and isinstance(backbone.classifier, nn.Sequential):
            if isinstance(backbone.classifier[-1], nn.Linear):
                backbone_out_features = backbone.classifier[-1].in_features
                self.backbone.classifier = nn.Identity()

        self.embedding_layer = nn.Linear(backbone_out_features, embedding_dim, bias=False)
        self.bn_embedding = nn.BatchNorm1d(embedding_dim)
        self.arcface_head = ArcFace(embedding_dim, num_classes, s, m)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        if features.dim() > 2:
            features = features.view(features.size(0), -1)

        embeddings = self.bn_embedding(self.embedding_layer(features))

        if labels is not None:
            return self.arcface_head(embeddings, labels)
        else:
            return embeddings

class FaceRecognitionDataset(Dataset):
    def __init__(self, img_dir, identity_df, person_to_label_map, transform=None):
        self.img_dir = img_dir
        self.identity_df = identity_df.copy()
        self.transform = transform
        self.person_to_label_map = person_to_label_map

        self.identity_df['label'] = self.identity_df['person_id'].map(self.person_to_label_map)
        self.image_names = self.identity_df['image_id'].tolist()
        self.labels = self.identity_df['label'].tolist()

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# пути
base_path = '/content/drive/MyDrive/celeba_project'
split_dataset_base_path_overlap = os.path.join(base_path, 'split_aligned_dataset_OVERLAP')

train_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'train_images_OVERLAP')
val_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'val_images_OVERLAP')
test_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'test_images_OVERLAP')

train_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'train_meta_OVERLAP.csv'))
val_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'val_meta_OVERLAP.csv'))
test_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'test_meta_OVERLAP.csv'))

all_person_ids_overlap = pd.concat([train_meta_df_overlap['person_id'], val_meta_df_overlap['person_id'], test_meta_df_overlap['person_id']]).unique()
person_to_label_map_overlap = {person_id: i for i, person_id in enumerate(all_person_ids_overlap)}
num_classes_overlap = len(person_to_label_map_overlap)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 32
train_dataset_overlap = FaceRecognitionDataset(
    img_dir=train_img_dir_overlap,
    identity_df=train_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

val_dataset_overlap = FaceRecognitionDataset(
    img_dir=val_img_dir_overlap,
    identity_df=val_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

test_dataset_overlap = FaceRecognitionDataset(
    img_dir=test_img_dir_overlap,
    identity_df=test_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

train_loader_overlap = DataLoader(train_dataset_overlap, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader_overlap = DataLoader(val_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader_overlap = DataLoader(test_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Number of unique persons (classes) in OVERLAP split: {num_classes_overlap}")
print(f"Train (OVERLAP) dataset size: {len(train_dataset_overlap)} images")
print(f"Validation (OVERLAP) dataset size: {len(val_dataset_overlap)} images")
print(f"Test (OVERLAP) dataset size: {len(test_dataset_overlap)} images")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

embedding_dim = 512
s_arcface = 64.0
m_arcface = 0.60

model_arcface = FaceRecognitionModelArcFace(
    num_classes=num_classes_overlap,
    backbone=model_resnet50,
    embedding_dim=embedding_dim,
    s=s_arcface,
    m=m_arcface
).to(device)

criterion_arcface = nn.CrossEntropyLoss()
optimizer_arcface = optim.Adam(model_arcface.parameters(), lr=1e-4)
scheduler_arcface = optim.lr_scheduler.StepLR(optimizer_arcface, step_size=5, gamma=0.5)

num_epochs_arcface = 20
best_val_accuracy_arcface = 0.0
model_save_path_arcface = os.path.join(base_path, 'face_recognition_resnet50_arcface_overlap_retrained.pth')

print("All data loaders and model initialized. Starting training")
print("\nStarting ArcFace training with adjusted parameters")

for epoch in range(num_epochs_arcface):
    model_arcface.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in tqdm(train_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs_arcface} (Train ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)

        optimizer_arcface.zero_grad()
        outputs = model_arcface(images, labels) # Pass labels to ArcFace head to get modified logits
        loss = criterion_arcface(outputs, labels)
        loss.backward()
        optimizer_arcface.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader_overlap)
    train_accuracy = correct_train / total_train

    # validation
    model_arcface.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs_arcface} (Validation ArcFace OVERLAP)"):
            images, labels = images.to(device), labels.to(device)
            outputs = model_arcface(images, labels)
            loss = criterion_arcface(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss /= len(val_loader_overlap)
    val_accuracy = correct_val / total_val

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

    # сохраняем лучшую модель
    if val_accuracy > best_val_accuracy_arcface:
        best_val_accuracy_arcface = val_accuracy
        torch.save(model_arcface.state_dict(), model_save_path_arcface)
        print(f"Model saved! Best validation accuracy: {best_val_accuracy_arcface:.4f}")

    scheduler_arcface.step()

print("\nArcFace training with adjusted parameters finished.")
print(f"Best validation accuracy achieved: {best_val_accuracy_arcface:.4f}")
print(f"Model with best validation accuracy saved to: {model_save_path_arcface}")

# test
model_arcface.load_state_dict(torch.load(model_save_path_arcface))
model_arcface.eval()

correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader_overlap, desc="Evaluating on Test Set (ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_arcface(images, labels)
        _, predicted = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_accuracy_arcface = correct_test / total_test
print(f"\nTest Accuracy with ArcFace Loss (OVERLAP dataset): {test_accuracy_arcface:.4f}")

Number of unique persons (classes) in OVERLAP split: 350
Train (OVERLAP) dataset size: 7350 images
Validation (OVERLAP) dataset size: 1575 images
Test (OVERLAP) dataset size: 1575 images
Using device: cuda
All data loaders and model initialized. Starting training...

Starting ArcFace training with adjusted parameters...


Epoch 1/20 (Validation ArcFace OVERLAP): 100%|██████████| 50/50 [00:11<00:00,  4.36it/s]


Epoch 1: Train Loss: 41.4621, Train Acc: 0.0000, Val Loss: 39.4699, Val Acc: 0.0000


Epoch 2/20 (Validation ArcFace OVERLAP): 100%|██████████| 50/50 [00:13<00:00,  3.81it/s]


Epoch 2: Train Loss: 35.4620, Train Acc: 0.0012, Val Loss: 36.6489, Val Acc: 0.0032
Model saved! Best validation accuracy: 0.0032


...

Epoch 19: Train Loss: 0.0027, Train Acc: 0.9996, Val Loss: 20.3486, Val Acc: 0.1867


Epoch 20/20 (Validation ArcFace OVERLAP): 100%|██████████| 50/50 [00:12<00:00,  4.04it/s]


Epoch 20: Train Loss: 0.0032, Train Acc: 0.9999, Val Loss: 20.3205, Val Acc: 0.1905
Model saved! Best validation accuracy: 0.1905

ArcFace training with adjusted parameters finished.
Best validation accuracy achieved: 0.1905
Model with best validation accuracy saved to: /content/drive/MyDrive/celeba_project/face_recognition_resnet50_arcface_overlap_retrained.pth


Evaluating on Test Set (ArcFace OVERLAP): 100%|██████████| 50/50 [04:52<00:00,  5.84s/it]


Test Accuracy with ArcFace Loss (OVERLAP dataset): 0.1619


немножко меньше требуемых 0.7 ...

In [ ]:
# опять местная версия улучшенная

В общем лучшее, что я смог выжать из ResNet50 это 0.4279 - см. ниже(причем увеличением количества эпох тут, похоже, делу не поможешь - несколько последних эпох accuracy на validation уже не росла)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import math
import torch.optim as optim
from google.colab import drive

# drive.mount('/content/drive')

class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_features, label):
        input_features = F.normalize(input_features)
        weight = F.normalize(self.weight)
        cosine = F.linear(input_features, weight)
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)

        output_logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output_logits *= self.s
        return output_logits

class FaceRecognitionModelArcFaceImproved(nn.Module):
    def __init__(self, num_classes, backbone, embedding_dim=512, s=64.0, m=0.50):
        super().__init__()
        self.backbone = backbone
        backbone_out_features = backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.embedding_layer = nn.Sequential(
            nn.Linear(backbone_out_features, 1024),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(1024, embedding_dim)
        )

        self.bn_embedding = nn.BatchNorm1d(embedding_dim)
        self.arcface_head = ArcFace(embedding_dim, num_classes, s, m)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        if features.dim() > 2:
            features = features.view(features.size(0), -1)

        embeddings = self.embedding_layer(features)
        embeddings = self.bn_embedding(embeddings)

        if labels is not None:
            return self.arcface_head(embeddings, labels)
        else:
            return embeddings


# пути
base_path = '/content/drive/MyDrive/celeba_project'
split_dataset_base_path_overlap = os.path.join(base_path, 'split_aligned_dataset_OVERLAP')

train_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'train_images_OVERLAP')
val_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'val_images_OVERLAP')
test_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'test_images_OVERLAP')

train_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'train_meta_OVERLAP.csv'))
val_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'val_meta_OVERLAP.csv'))
test_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'test_meta_OVERLAP.csv'))

all_person_ids_overlap = pd.concat([train_meta_df_overlap['person_id'], val_meta_df_overlap['person_id'], test_meta_df_overlap['person_id']]).unique()
person_to_label_map_overlap = {person_id: i for i, person_id in enumerate(all_person_ids_overlap)}
num_classes_overlap = len(person_to_label_map_overlap)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FaceRecognitionDataset(Dataset):
    def __init__(self, img_dir, identity_df, person_to_label_map, transform=None):
        self.img_dir = img_dir
        self.identity_df = identity_df.copy()
        self.transform = transform
        self.person_to_label_map = person_to_label_map

        self.identity_df['label'] = self.identity_df['person_id'].map(self.person_to_label_map)
        self.image_names = self.identity_df['image_id'].tolist()
        self.labels = self.identity_df['label'].tolist()

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

batch_size = 32
train_dataset_overlap = FaceRecognitionDataset(
    img_dir=train_img_dir_overlap,
    identity_df=train_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

val_dataset_overlap = FaceRecognitionDataset(
    img_dir=val_img_dir_overlap,
    identity_df=val_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

test_dataset_overlap = FaceRecognitionDataset(
    img_dir=test_img_dir_overlap,
    identity_df=test_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform
)

train_loader_overlap = DataLoader(train_dataset_overlap, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader_overlap = DataLoader(val_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader_overlap = DataLoader(test_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Number of unique persons (classes) in OVERLAP split: {num_classes_overlap}")
print(f"Train (OVERLAP) dataset size: {len(train_dataset_overlap)} images")
print(f"Validation (OVERLAP) dataset size: {len(val_dataset_overlap)} images")
print(f"Test (OVERLAP) dataset size: {len(test_dataset_overlap)} images")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

embedding_dim = 512
s_arcface = 64.0
m_arcface = 0.60

model_arcface = FaceRecognitionModelArcFaceImproved(
    num_classes=num_classes_overlap,
    backbone=model_resnet50,
    embedding_dim=embedding_dim,
    s=s_arcface,
    m=m_arcface
).to(device)

criterion_arcface = nn.CrossEntropyLoss()
optimizer_arcface = optim.Adam(model_arcface.parameters(), lr=1e-4)

num_epochs = 20
best_val_accuracy_arcface = 0.0
model_save_path_arcface = os.path.join(base_path, 'face_recognition_resnet50_arcface_overlap.pth')

print("\nStarting training with ArcFace Loss on OVERLAP dataset")

for epoch in range(num_epochs):
    model_arcface.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in tqdm(train_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Train ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)

        optimizer_arcface.zero_grad()
        outputs = model_arcface(images, labels)
        loss = criterion_arcface(outputs, labels)
        loss.backward()
        optimizer_arcface.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader_overlap)
    train_accuracy = correct_train / total_train

    # validation
    model_arcface.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Validation ArcFace OVERLAP)"):
            images, labels = images.to(device), labels.to(device)
            outputs = model_arcface(images, labels)
            loss = criterion_arcface(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss /= len(val_loader_overlap)
    val_accuracy = correct_val / total_val

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

    # сохраняем лучшую модель
    if val_accuracy > best_val_accuracy_arcface:
        best_val_accuracy_arcface = val_accuracy
        torch.save(model_arcface.state_dict(), model_save_path_arcface)
        print(f"Model saved! Best validation accuracy: {best_val_accuracy_arcface:.4f}")

print("\nTraining with ArcFace Loss on OVERLAP dataset finished.")
print(f"Best validation accuracy achieved: {best_val_accuracy_arcface:.4f}")
print(f"Model with best validation accuracy saved to: {model_save_path_arcface}")

# test
model_arcface.load_state_dict(torch.load(model_save_path_arcface))
model_arcface.eval()

correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader_overlap, desc="Evaluating on Test Set (ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_arcface(images, labels)
        _, predicted = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_accuracy_arcface = correct_test / total_test
print(f"\nTest Accuracy with ArcFace Loss (OVERLAP dataset): {test_accuracy_arcface:.4f}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of unique persons (classes) in OVERLAP split: 350
Train (OVERLAP) dataset size: 7350 images
Validation (OVERLAP) dataset size: 1575 images
Test (OVERLAP) dataset size: 1575 images
Using device: cuda

Starting training with ArcFace Loss on OVERLAP dataset...


Epoch 1/20 (Validation ArcFace OVERLAP): 100%|██████████| 50/50 [05:15<00:00,  6.31s/it]


Epoch 1: Train Loss: 43.4615, Train Acc: 0.0000, Val Loss: 40.6585, Val Acc: 0.0000


Epoch 2/20 (Validation ArcFace OVERLAP): 100%|██████████| 50/50 [00:10<00:00,  4.78it/s]


Epoch 2: Train Loss: 40.0649, Train Acc: 0.0000, Val Loss: 39.0050, Val Acc: 0.0000


...

Epoch 19: Train Loss: 0.6538, Train Acc: 0.8576, Val Loss: 19.4420, Val Acc: 0.4006


Epoch 20/20 (Validation ArcFace OVERLAP): 100%|██████████| 50/50 [00:12<00:00,  4.12it/s]


Epoch 20: Train Loss: 0.7654, Train Acc: 0.8386, Val Loss: 20.9494, Val Acc: 0.3746

Training with ArcFace Loss on OVERLAP dataset finished.
Best validation accuracy achieved: 0.4368
Model with best validation accuracy saved to: /content/drive/MyDrive/celeba_project/face_recognition_resnet50_arcface_overlap.pth


Evaluating on Test Set (ArcFace OVERLAP): 100%|██████████| 50/50 [05:07<00:00,  6.16s/it]


Test Accuracy with ArcFace Loss (OVERLAP dataset): 0.4279


## ЭКСПЕРИМЕНТ с EfficientNet

Поскольку с помощью архитектуры ResNet50 мне не удалось достигнуть целевой точности в 70 %, попробуем заменить её на EfficientNet B0.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import math
import torch.optim as optim
from google.colab import drive

# drive.mount('/content/drive')

class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_features, label):
        input_features = F.normalize(input_features)
        weight = F.normalize(self.weight)
        cosine = F.linear(input_features, weight)
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)

        output_logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output_logits *= self.s
        return output_logits

class FaceRecognitionModelArcFaceEfficientNet(nn.Module):
    def __init__(self, num_classes, backbone, embedding_dim=512, s=64.0, m=0.50):
        super().__init__()
        self.backbone = backbone

        backbone_out_features = None
        if hasattr(backbone, 'fc') and isinstance(backbone.fc, nn.Linear):
            backbone_out_features = backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif hasattr(backbone, 'classifier') and isinstance(backbone.classifier, nn.Sequential):
            if isinstance(backbone.classifier[-1], nn.Linear):
                backbone_out_features = backbone.classifier[-1].in_features
                self.backbone.classifier = nn.Identity()
        else:
            raise ValueError("Could not determine backbone output features.")

        if backbone_out_features is None:
            raise ValueError("Backbone output features could not be determined after checking 'fc' and 'classifier'.")

        self.embedding_layer = nn.Sequential(
            nn.Linear(backbone_out_features, 1024),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(1024, embedding_dim)
        )

        self.bn_embedding = nn.BatchNorm1d(embedding_dim)
        self.arcface_head = ArcFace(embedding_dim, num_classes, s, m)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        if features.dim() > 2:
            features = features.view(features.size(0), -1)

        embeddings = self.embedding_layer(features)
        embeddings = self.bn_embedding(embeddings)

        if labels is not None:
            return self.arcface_head(embeddings, labels)
        else:
            return embeddings

class FaceRecognitionDataset(Dataset):
    def __init__(self, img_dir, identity_df, person_to_label_map, transform=None):
        self.img_dir = img_dir
        self.identity_df = identity_df.copy()
        self.transform = transform
        self.person_to_label_map = person_to_label_map

        self.identity_df['label'] = self.identity_df['person_id'].map(self.person_to_label_map)
        self.image_names = self.identity_df['image_id'].tolist()
        self.labels = self.identity_df['label'].tolist()

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# пути
base_path = '/content/drive/MyDrive/celeba_project'
split_dataset_base_path_overlap = os.path.join(base_path, 'split_aligned_dataset_OVERLAP')

train_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'train_images_OVERLAP')
val_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'val_images_OVERLAP')
test_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'test_images_OVERLAP')

train_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'train_meta_OVERLAP.csv'))
val_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'val_meta_OVERLAP.csv'))
test_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'test_meta_OVERLAP.csv'))

all_person_ids_overlap = pd.concat([train_meta_df_overlap['person_id'], val_meta_df_overlap['person_id'], test_meta_df_overlap['person_id']]).unique()
person_to_label_map_overlap = {person_id: i for i, person_id in enumerate(all_person_ids_overlap)}
num_classes_overlap = len(person_to_label_map_overlap)

transform_efficientnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 32
train_dataset_overlap = FaceRecognitionDataset(
    img_dir=train_img_dir_overlap,
    identity_df=train_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform_efficientnet
)

val_dataset_overlap = FaceRecognitionDataset(
    img_dir=val_img_dir_overlap,
    identity_df=val_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform_efficientnet
)

test_dataset_overlap = FaceRecognitionDataset(
    img_dir=test_img_dir_overlap,
    identity_df=test_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform_efficientnet
)

train_loader_overlap = DataLoader(train_dataset_overlap, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader_overlap = DataLoader(val_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader_overlap = DataLoader(test_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Number of unique persons (classes) in OVERLAP split: {num_classes_overlap}")
print(f"Train (OVERLAP) dataset size: {len(train_dataset_overlap)} images")
print(f"Validation (OVERLAP) dataset size: {len(val_dataset_overlap)} images")
print(f"Test (OVERLAP) dataset size: {len(test_dataset_overlap)} images")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_efficientnet_b0 = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

embedding_dim = 512
s_arcface = 32.0
m_arcface = 0.60

model_arcface_efficientnet = FaceRecognitionModelArcFaceEfficientNet(
    num_classes=num_classes_overlap,
    backbone=model_efficientnet_b0,
    embedding_dim=embedding_dim,
    s=s_arcface,
    m=m_arcface
).to(device)

criterion_arcface = nn.CrossEntropyLoss()
optimizer_arcface = optim.Adam(model_arcface_efficientnet.parameters(), lr=1e-3)

num_epochs = 20
best_val_accuracy_arcface_efficientnet = 0.0
model_save_path_arcface_efficientnet = os.path.join(base_path, 'face_recognition_efficientnet_arcface_overlap.pth')

print("\nStarting training with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset")

for epoch in range(num_epochs):
    model_arcface_efficientnet.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in tqdm(train_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Train EfficientNet ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)

        optimizer_arcface.zero_grad()
        outputs = model_arcface_efficientnet(images, labels)
        loss = criterion_arcface(outputs, labels)
        loss.backward()
        optimizer_arcface.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader_overlap)
    train_accuracy = correct_train / total_train

    # validation
    model_arcface_efficientnet.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Validation EfficientNet ArcFace OVERLAP)"):
            images, labels = images.to(device), labels.to(device)
            outputs = model_arcface_efficientnet(images, labels)
            loss = criterion_arcface(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss /= len(val_loader_overlap)
    val_accuracy = correct_val / total_val

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

    if val_accuracy > best_val_accuracy_arcface_efficientnet:
        best_val_accuracy_arcface_efficientnet = val_accuracy
        torch.save(model_arcface_efficientnet.state_dict(), model_save_path_arcface_efficientnet)
        print(f"Model saved! Best validation accuracy: {best_val_accuracy_arcface_efficientnet:.4f}")

print("\nTraining with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset finished.")
print(f"Best validation accuracy achieved: {best_val_accuracy_arcface_efficientnet:.4f}")
print(f"Model with best validation accuracy saved to: {model_save_path_arcface_efficientnet}")

# test
model_arcface_efficientnet.load_state_dict(torch.load(model_save_path_arcface_efficientnet))
model_arcface_efficientnet.eval()

correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader_overlap, desc="Evaluating on Test Set (EfficientNet ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_arcface_efficientnet(images, labels)
        _, predicted = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_accuracy_arcface_efficientnet = correct_test / total_test
print(f"\nTest Accuracy with EfficientNet B0 backbone and ArcFace Loss (OVERLAP dataset): {test_accuracy_arcface_efficientnet:.4f}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Number of unique persons (classes) in OVERLAP split: 350
Train (OVERLAP) dataset size: 7350 images
Validation (OVERLAP) dataset size: 1575 images
Test (OVERLAP) dataset size: 1575 images
Using device: cuda

Starting training with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset...


Epoch 1/20 (Train EfficientNet ArcFace OVERLAP):   0%|          | 0/230 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 1/20 (Train EfficientNet ArcFace OVERLAP): 100%|██████████| 230/230 [08:10<00:00,  2.13s/it]
Epoch 1/20 (Validation EfficientNet ArcFace OVERLAP): 100%|██████████| 50/50 [02:14<00:00,  2.68s/it]


Epoch 1: Train Loss: 23.6671, Train Acc: 0.0000, Val Loss: 22.6134, Val Acc: 0.0000


Epoch 2/20 (Train EfficientNet ArcFace OVERLAP): 100%|██████████| 230/230 [00:57<00:00,  4.00it/s]
Epoch 2/20 (Validation EfficientNet ArcFace OVERLAP): 100%|██████████| 50/50 [00:10<00:00,  4.56it/s]


...

Epoch 19: Train Loss: 1.1957, Train Acc: 0.7869, Val Loss: 6.2243, Val Acc: 0.6260


Epoch 20/20 (Train EfficientNet ArcFace OVERLAP): 100%|██████████| 230/230 [01:00<00:00,  3.81it/s]
Epoch 20/20 (Validation EfficientNet ArcFace OVERLAP): 100%|██████████| 50/50 [00:11<00:00,  4.34it/s]


Epoch 20: Train Loss: 0.9535, Train Acc: 0.8272, Val Loss: 6.4785, Val Acc: 0.6406

Training with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset finished.
Best validation accuracy achieved: 0.6406
Model with best validation accuracy saved to: /content/drive/MyDrive/celeba_project/face_recognition_efficientnet_arcface_overlap.pth


Evaluating on Test Set (EfficientNet ArcFace OVERLAP): 100%|██████████| 50/50 [02:17<00:00,  2.75s/it]


Test Accuracy with EfficientNet B0 backbone and ArcFace Loss (OVERLAP dataset): 0.6337


После добавления аугментации с помощью `RandomRotation` и `ColorJitter` и введения планировщик скорости обучения `ReduceLROnPlateau`, который адаптивно снижает скорость обучения, когда потери валидации перестают улучшаться, на 30 эпохах было получено: на validation 0.6908; на тесте 0.6857
Уже почти.
На 50 эпохах:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import math
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from google.colab import drive

# drive.mount('/content/drive')

class ArcFace(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_features, label):
        input_features = F.normalize(input_features)
        weight = F.normalize(self.weight)

        cosine = F.linear(input_features, weight)

        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m

        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)

        output_logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output_logits *= self.s

        return output_logits

class FaceRecognitionModelArcFaceEfficientNet(nn.Module):
    def __init__(self, num_classes, backbone, embedding_dim=512, s=64.0, m=0.50):
        super().__init__()
        self.backbone = backbone

        backbone_out_features = None
        if hasattr(backbone, 'fc') and isinstance(backbone.fc, nn.Linear):
            backbone_out_features = backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif hasattr(backbone, 'classifier') and isinstance(backbone.classifier, nn.Sequential):
            if isinstance(backbone.classifier[-1], nn.Linear):
                backbone_out_features = backbone.classifier[-1].in_features
                self.backbone.classifier = nn.Identity()
        else:
            raise ValueError("Could not determine backbone output features.")

        if backbone_out_features is None:
            raise ValueError("Backbone output features could not be determined after checking 'fc' and 'classifier'.")

        self.embedding_layer = nn.Sequential(
            nn.Linear(backbone_out_features, 1024),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(1024, embedding_dim)
        )

        self.bn_embedding = nn.BatchNorm1d(embedding_dim)
        self.arcface_head = ArcFace(embedding_dim, num_classes, s, m)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        if features.dim() > 2:
            features = features.view(features.size(0), -1)

        embeddings = self.embedding_layer(features)
        embeddings = self.bn_embedding(embeddings)

        if labels is not None:
            return self.arcface_head(embeddings, labels)
        else:
            return embeddings

class FaceRecognitionDataset(Dataset):
    def __init__(self, img_dir, identity_df, person_to_label_map, transform=None):
        self.img_dir = img_dir
        self.identity_df = identity_df.copy()
        self.transform = transform
        self.person_to_label_map = person_to_label_map

        self.identity_df['label'] = self.identity_df['person_id'].map(self.person_to_label_map)
        self.image_names = self.identity_df['image_id'].tolist()
        self.labels = self.identity_df['label'].tolist()

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# пути
base_path = '/content/drive/MyDrive/celeba_project'
split_dataset_base_path_overlap = os.path.join(base_path, 'split_aligned_dataset_OVERLAP')

train_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'train_images_OVERLAP')
val_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'val_images_OVERLAP')
test_img_dir_overlap = os.path.join(split_dataset_base_path_overlap, 'test_images_OVERLAP')

train_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'train_meta_OVERLAP.csv'))
val_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'val_meta_OVERLAP.csv'))
test_meta_df_overlap = pd.read_csv(os.path.join(split_dataset_base_path_overlap, 'test_meta_OVERLAP.csv'))

all_person_ids_overlap = pd.concat([train_meta_df_overlap['person_id'], val_meta_df_overlap['person_id'], test_meta_df_overlap['person_id']]).unique()
person_to_label_map_overlap = {person_id: i for i, person_id in enumerate(all_person_ids_overlap)}
num_classes_overlap = len(person_to_label_map_overlap)

transform_enhanced = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 32
train_dataset_overlap = FaceRecognitionDataset(
    img_dir=train_img_dir_overlap,
    identity_df=train_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform_enhanced
)

val_dataset_overlap = FaceRecognitionDataset(
    img_dir=val_img_dir_overlap,
    identity_df=val_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform_eval
)

test_dataset_overlap = FaceRecognitionDataset(
    img_dir=test_img_dir_overlap,
    identity_df=test_meta_df_overlap,
    person_to_label_map=person_to_label_map_overlap,
    transform=transform_eval
)

train_loader_overlap = DataLoader(train_dataset_overlap, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader_overlap = DataLoader(val_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader_overlap = DataLoader(test_dataset_overlap, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Number of unique persons (classes) in OVERLAP split: {num_classes_overlap}")
print(f"Train (OVERLAP) dataset size: {len(train_dataset_overlap)} images")
print(f"Validation (OVERLAP) dataset size: {len(val_dataset_overlap)} images")
print(f"Test (OVERLAP) dataset size: {len(test_dataset_overlap)} images")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_efficientnet_b0 = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

embedding_dim = 512
s_arcface = 32.0
m_arcface = 0.70  # увеличил с 0.6

model_arcface_efficientnet_tuned = FaceRecognitionModelArcFaceEfficientNet(
    num_classes=num_classes_overlap,
    backbone=model_efficientnet_b0,
    embedding_dim=embedding_dim,
    s=s_arcface,
    m=m_arcface
).to(device)

criterion_arcface = nn.CrossEntropyLoss()
optimizer_arcface = optim.Adam(model_arcface_efficientnet_tuned.parameters(), lr=5e-4)
scheduler_arcface = ReduceLROnPlateau(optimizer_arcface, mode='max', factor=0.5, patience=5)

num_epochs = 50 # увеличил с 30 до 50
best_val_accuracy_arcface_efficientnet_tuned = 0.0
model_save_path_arcface_efficientnet_tuned = os.path.join(base_path, 'face_recognition_efficientnet_arcface_overlap_tuned.pth')

print("\nStarting training with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset (Further Tuned)")

for epoch in range(num_epochs):
    model_arcface_efficientnet_tuned.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in tqdm(train_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Train EfficientNet ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)

        optimizer_arcface.zero_grad()
        outputs = model_arcface_efficientnet_tuned(images, labels)
        loss = criterion_arcface(outputs, labels)
        loss.backward()
        optimizer_arcface.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader_overlap)
    train_accuracy = correct_train / total_train

    # validation
    model_arcface_efficientnet_tuned.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader_overlap, desc=f"Epoch {epoch+1}/{num_epochs} (Validation EfficientNet ArcFace OVERLAP)"):
            images, labels = images.to(device), labels.to(device)
            outputs = model_arcface_efficientnet_tuned(images, labels)
            loss = criterion_arcface(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss /= len(val_loader_overlap)
    val_accuracy = correct_val / total_val

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

    scheduler_arcface.step(val_accuracy)

    if val_accuracy > best_val_accuracy_arcface_efficientnet_tuned:
        best_val_accuracy_arcface_efficientnet_tuned = val_accuracy
        torch.save(model_arcface_efficientnet_tuned.state_dict(), model_save_path_arcface_efficientnet_tuned)
        print(f"Model saved! Best validation accuracy: {best_val_accuracy_arcface_efficientnet_tuned:.4f}")

print("\nTraining with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset (Further Tuned) finished.")
print(f"Best validation accuracy achieved: {best_val_accuracy_arcface_efficientnet_tuned:.4f}")
print(f"Model with best validation accuracy saved to: {model_save_path_arcface_efficientnet_tuned}")

# test
model_arcface_efficientnet_tuned.load_state_dict(torch.load(model_save_path_arcface_efficientnet_tuned))
model_arcface_efficientnet_tuned.eval()

correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader_overlap, desc="Evaluating on Test Set (EfficientNet ArcFace OVERLAP)"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_arcface_efficientnet_tuned(images, labels)
        _, predicted = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_accuracy_arcface_efficientnet_tuned = correct_test / total_test
print(f"\nTest Accuracy with EfficientNet B0 backbone and ArcFace Loss (OVERLAP dataset, Further Tuned): {test_accuracy_arcface_efficientnet_tuned:.4f}")

Number of unique persons (classes) in OVERLAP split: 350
Train (OVERLAP) dataset size: 7350 images
Validation (OVERLAP) dataset size: 1575 images
Test (OVERLAP) dataset size: 1575 images
Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 159MB/s]



Starting training with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset (Further Tuned)...


Epoch 1/50 (Train EfficientNet ArcFace OVERLAP): 100%|██████████| 230/230 [21:45<00:00,  5.67s/it]
Epoch 1/50 (Validation EfficientNet ArcFace OVERLAP): 100%|██████████| 50/50 [04:34<00:00,  5.49s/it]


Epoch 1: Train Loss: 26.3649, Train Acc: 0.0000, Val Loss: 24.9805, Val Acc: 0.0000


...

Epoch 49: Train Loss: 0.1728, Train Acc: 0.9778, Val Loss: 4.7440, Val Acc: 0.8063
Model saved! Best validation accuracy: 0.8063


Epoch 50/50 (Train EfficientNet ArcFace OVERLAP): 100%|██████████| 230/230 [01:36<00:00,  2.38it/s]
Epoch 50/50 (Validation EfficientNet ArcFace OVERLAP): 100%|██████████| 50/50 [00:11<00:00,  4.27it/s]


Epoch 50: Train Loss: 0.1501, Train Acc: 0.9830, Val Loss: 4.6096, Val Acc: 0.8108
Model saved! Best validation accuracy: 0.8108

Training with ArcFace Loss and EfficientNet B0 backbone on OVERLAP dataset (Further Tuned) finished.
Best validation accuracy achieved: 0.8108
Model with best validation accuracy saved to: /content/drive/MyDrive/celeba_project/face_recognition_efficientnet_arcface_overlap_tuned.pth


Evaluating on Test Set (EfficientNet ArcFace OVERLAP): 100%|██████████| 50/50 [04:38<00:00,  5.58s/it]


Test Accuracy with EfficientNet B0 backbone and ArcFace Loss (OVERLAP dataset, Further Tuned): 0.8108


Если сравнивать  CE loss и ArcFace loss, то, если говорить кратко: CE учит модель разделять классы, а ArcFace учит модель сжимать один класс и раздвигать разные классы на гиперсфере.
Иишка мне такую вот кратенькую понятную табличку сравнительную предложил:

### Сравнение CE Loss и ArcFace Loss

| Характеристика | Cross-Entropy (CE) Loss | ArcFace (Angular Margin) Loss |
| :--- | :--- | :--- |
| **Цель** | Максимизация вероятности правильного класса. | Максимизация углового расстояния между классами. |
| **Пространство** | Евклидово пространство логитов. | Гиперсфера (нормализованные векторы). |
| **Внутриклассовая компактность** | Не гарантируется (фокус только на границе). | Высокая (благодаря штрафу $m$). |
| **Межклассовая разделимость** | Достаточная для классификации. | Экстремальная (для задач верификации/метрического обучения). |
| **Сложность внедрения** | Стандартный слой `nn.Linear`. | Кастомный слой с нормализацией и margin. |

Что касается непосредственно выводов по данному ноутбуку: обучение на ArcFace занимает больше машинного времени и на ResNet50 мне вообще не удалось добиться поставленного минимума accuracy в 70%.
Под конец я понял, что на начальном этапе допустил косяк - на этапе формирования "выборочного" датасета - не отслеживал, сколько туда попадает персон женского, а сколько мужского пола (открывал, листал фотки изначального "сырого" датасет - показалось, что их там примерно одинаково).
А получилось вот:

In [ ]:
import pandas as pd
import os

new_dataset_image_ids = processed_landmarks_df_new['image_id'].tolist()
gender_df = attr_df[attr_df.index.isin(new_dataset_image_ids)].copy()
gender_df['Gender'] = gender_df['Male'].apply(lambda x: 'Male' if x == 1 else 'Female')
gender_counts = gender_df['Gender'].value_counts()
gender_percentages = gender_df['Gender'].value_counts(normalize=True) * 100

print("Gender distribution in rotated_aligned_dataset_NEW:")
print(gender_counts)
print("\nPercentage distribution:")
print(gender_percentages.round(2).astype(str) + '%')

Gender distribution in rotated_aligned_dataset_NEW:
Gender
Female    7920
Male      2580
Name: count, dtype: int64

Percentage distribution:
Gender
Female    75.43%
Male      24.57%
Name: proportion, dtype: object


Не знаю на сколько это критично... но перекос в сторону прекрасного пола имеет место быть